# CycleGAN melon: EMA of generator weights, stability test

Two full 60+60 runs with mixed precision, in parallel on GPU T4 x2 (one per GPU;
the proxy session showed Kaggle bills the session, not each GPU):

* `G0-EMA`: the G0 objective on the 302-crop pool.
* `G0X-EMA`: the G0X-v2 package on its curated pool.

Each run keeps an exponential moving average of its generator weights
(decay 0.999, Yazici et al., ICLR 2019) beside the raw weights. EMA does not
change training, so raw and EMA checkpoints of one run form a paired comparison.
The pre-registered criteria are in `Ujicoba CycleGAN/ABLASI_HEMAT.md`, section
"Uji EMA". Candidates stay `needs_review`. Validation and test splits are never opened.

In [ ]:
# The input mounts are read only and ultralytics writes next to its working
# directory, so the whole code tree is copied into /kaggle/working first.
import os, shutil, subprocess, sys, time
from pathlib import Path

INPUT = Path('/kaggle/input')
print('mounted:', sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else 'nothing')

def find(*names, marker=None):
    # Kaggle mounts a dataset at /kaggle/input/<slug> or, when several are
    # attached, at /kaggle/input/datasets/<owner>/<slug>, so the slug is searched
    # by name at any depth before falling back to the marker file.
    for name in names:
        for candidate in [INPUT / name, *sorted(INPUT.rglob(name))]:
            if candidate.is_dir() and (marker is None or (candidate / marker).exists()):
                return candidate
    hits = sorted({p.parent for p in INPUT.rglob(marker)}) if marker else []
    if len(hits) == 1: return hits[0]
    raise SystemExit(f'cannot locate {names} (marker {marker}); found {hits}')

SOURCE = find('melon-cyclegan-code', marker='Ujicoba CycleGAN')
DATASET = find('melon-dataset-regrouped-v4', 'melon-final-reviewed-v4', marker='manifest.csv')
ASSETS = find('melon-benchmark-assets', marker='reviewed_masks.json')

WORK = Path('/kaggle/working')
CODE = WORK / 'code'
if CODE.exists(): shutil.rmtree(CODE)
shutil.copytree(SOURCE, CODE)
HERE = CODE / 'Ujicoba CycleGAN'
os.environ['MELON_DATASET'] = str(DATASET)
os.environ['MELON_ASSETS'] = str(ASSETS)
sys.path.insert(0, str(HERE))

import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('code:', sorted(p.name for p in HERE.iterdir()))
print('dataset:', (DATASET / 'manifest.csv').exists(), 'assets:', (ASSETS / 'reviewed_masks.json').exists())


def run(*command, cwd=None):
    started = time.time()
    result = subprocess.run([sys.executable, *[str(c) for c in command]], cwd=str(cwd or HERE))
    print(f'-> returncode {result.returncode} in {time.time() - started:.1f}s', flush=True)
    if result.returncode != 0: raise SystemExit(result.returncode)

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-t', '.'],
               cwd=str(HERE), env={**os.environ, 'CUDA_VISIBLE_DEVICES': '',
                                   'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2'}, check=True)

In [ ]:
CROPS = WORK / 'crops256'
if not (CROPS / 'crops_manifest.json').exists():
    run(HERE / 'tools' / 'build_crops.py', '--out', CROPS, '--dataset', DATASET)
import json
manifest = json.loads((CROPS / 'crops_manifest.json').read_text())
print(manifest['counts'], manifest['mask_sources'])
print('crops_digest', manifest['crops_digest'])
assert manifest['crops_digest'] == '82e8ae0980142fdebb3aa443e14ce068b60247bbdb1aac047a7bfba1c77d075a', 'crop set changed'
print((CROPS / 'color_centroids.json').read_text())

In [ ]:
# Both v2 generators must use exactly the same image and mask bytes.
SOURCE_CROPS = CROPS
CROPS = WORK / 'crops256_g0x_v2'
run(HERE / 'tools' / 'prepare_g0x_data.py', '--source', SOURCE_CROPS, '--out', CROPS)
print((CROPS / 'color_margin.json').read_text())
# Preflight resolves the calibrated margin and checks every crop/mask hash.

In [ ]:
import torch, subprocess
CKPT = WORK / 'checkpoints'
BUDGET = {'melon_cyclegan_g0_ema': 240, 'melon_cyclegan_g0x_ema': 240}  # minutes, safety net only
JOBS = [('config/g0_ema.json', 'melon_cyclegan_g0_ema', SOURCE_CROPS),
        ('config/g0x_ema.json', 'melon_cyclegan_g0x_ema', CROPS)]
n_gpu = torch.cuda.device_count()
print('visible GPUs:', n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
def launch(config, name, pool, gpu):
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu), 'PYTHONPATH': str(HERE)}
    log = open(WORK / f'{name}.log', 'w')
    command = [sys.executable, str(HERE / 'tools' / 'train_cyclegan.py'), '--config', str(HERE / config),
               '--dataroot', str(pool), '--checkpoints-dir', str(CKPT), '--name', name,
               '--', '--time_budget_minutes', str(BUDGET[name])]
    return subprocess.Popen(command, cwd=str(HERE), env=env, stdout=log, stderr=subprocess.STDOUT)
started = time.time()
if n_gpu >= 2:
    procs = [launch(c, n, p, i) for i, (c, n, p) in enumerate(JOBS)]
    codes = [proc.wait() for proc in procs]
else:
    codes = [launch(c, n, p, 0).wait() for c, n, p in JOBS]
print('return codes', codes, 'wall minutes', round((time.time() - started) / 60, 1))
for _, name, _ in JOBS:
    print(name, (WORK / f'{name}.log').read_text()[-1500:])
if any(codes): raise SystemExit('a training failed; see the logs above')
for _, name, _ in JOBS:
    history = json.loads((CKPT / name / 'loss.json').read_text())
    print(name, 'seconds', history['seconds'], 'epochs', history['epochs'], 'stopped', history['stopped'])
    if history['stopped']: raise SystemExit(f'{name} did not finish its schedule')
    assert (CKPT / name / 'latest_net_G_A_ema.pth').exists(), 'EMA weights missing'

In [ ]:
# Sweeps of raw and EMA checkpoints, then one bundle per (run, weights) from the final checkpoint.
SWEEPS = {}
for _, name, pool in JOBS:
    for weights in ('raw', 'ema'):
        tag = f"{name.replace('melon_cyclegan_', '')}_{weights}"
        out = WORK / f'sweep_{tag}.json'
        run(HERE / 'tools' / 'checkpoint_sweep.py', '--checkpoints-dir', CKPT, '--name', name,
            '--crops', SOURCE_CROPS, '--dataset', DATASET, '--device', 'cuda',
            '--decay-start', '61', '--weights', weights, '--out', out)
        SWEEPS[tag] = json.loads(out.read_text())
        bundle = WORK / f'bundle_cyclegan_{tag}'
        run(HERE / 'tools' / 'generate_candidates.py', '--checkpoints-dir', CKPT, '--name', name,
            '--condition', f'cyclegan_{tag}', '--out', bundle, '--slots', '2', '--dataset', DATASET,
            '--weights', weights)
        qc_flags = ['--composite-trained'] if 'g0x' in name else []
        run(HERE / 'tools' / 'run_qc.py', '--bundle', bundle, '--crops', SOURCE_CROPS, '--dataset', DATASET, *qc_flags)
        run(HERE / 'tools' / 'review.py', 'export', '--bundle', bundle)
        run(HERE / 'tools' / 'review.py', 'tool', '--bundle', bundle)
print('sweeps done:', sorted(SWEEPS))

In [ ]:
# Keep generator weights every 10 epochs plus latest (raw and EMA); latest D only.
import re
shutil.rmtree(CODE, ignore_errors=True)
for pool in (CROPS, SOURCE_CROPS):
    for folder in ('trainA', 'trainB', 'trainA_mask', 'trainB_mask'):
        shutil.rmtree(pool / folder, ignore_errors=True)
for run_dir in sorted(CKPT.iterdir()):
    for path in run_dir.glob('*_net_*.pth'):
        m = re.match(r'(\d+)_net_', path.name)
        if not m: continue
        if '_net_D_' in path.name or int(m.group(1)) % 10: path.unlink()
print(sorted(p.name for p in WORK.iterdir()))